# Export the multi-class predicted probabilities per single cell

We apply the multi-class model (`model_all_hearts_final_multiclass`) to every single cell on the training plate and export the predicted probability of each class per cell.
Each cell has the identifying metadata (plate, well, site, and nuclei object number) so the probabilities can be merged with the normalized single-cell profiles for the plate.

The `dataset` column shows the split each cell is from.
Cells in the `train` split were used to train the model, so their probabilities are higher confidence than the cells the model has not seen (`test` and holdout splits).

## Import libraries

In [1]:
import os
import pathlib
import sys

import pandas as pd
from IPython.display import display
from joblib import load

sys.path.append("../utils")
from training_utils import get_X_y_data

## Set paths and variables

In [2]:
# Set the plate to process with the PLATE_TO_TRAIN environment variable (defaults to the subtyping plate)
plate_to_process = os.environ.get("PLATE_TO_TRAIN", "CARD-CelIns-CX7_260814100001")

model_path = pathlib.Path(
    f"./models/{plate_to_process}/model_all_hearts_final_multiclass.joblib"
).resolve(strict=True)
encoder_path = pathlib.Path(
    f"./encoder_results/{plate_to_process}/label_encoder_multi-class.joblib"
).resolve(strict=True)
split_dir = pathlib.Path(f"./data_splits/{plate_to_process}").resolve(strict=True)

# Normalized profiles for the plate (can be in a subfolder, e.g. hlhs/), used to check the cells match
normalized_path = next(
    pathlib.Path("../3.preprocessing_profiles/data/single_cell_profiles")
    .resolve(strict=True)
    .rglob(f"{plate_to_process}_sc_normalized.parquet")
)

# Output dir for the predicted probabilities
performance_metrics_dir = pathlib.Path(f"./performance_metrics/{plate_to_process}")
performance_metrics_dir.mkdir(parents=True, exist_ok=True)

# All splits (the female heart holdout is only made when the plate has the female hearts)
split_files = {
    "train": "training_split.parquet",
    "test": "testing_split.parquet",
    "holdout_well": "holdout_well_split.parquet",
    "holdout_female_hearts": "holdout_female_hearts_split.parquet",
}
split_files = {
    split_name: file_name
    for split_name, file_name in split_files.items()
    if (split_dir / file_name).exists()
}

# Columns that uniquely identify a single cell
cell_id_cols = [
    "Metadata_Plate",
    "Metadata_Well",
    "Metadata_Site",
    "Metadata_Nuclei_Number_Object_Number",
]
# Metadata to keep with the predictions
metadata_cols = cell_id_cols + [
    "Metadata_heart_number",
    "Metadata_cell_type",
    "Metadata_heart_failure_type",
    "Metadata_treatment",
]

# Healthy hearts have no heart failure type
label = "Metadata_heart_failure_type"
healthy_label = "Healthy"

## Predict the class probabilities for every single cell

The model classes are encoded, so we use the multi-class label encoder to get the class names.

In [3]:
model = load(model_path)
label_encoder = load(encoder_path)
class_names = label_encoder.inverse_transform(model.classes_)

cells_df = pd.concat(
    [
        pd.read_parquet(split_dir / file_name).assign(dataset=split_name)
        for split_name, file_name in split_files.items()
    ],
    ignore_index=True,
)

# Class of each cell (healthy hearts have no heart failure type, so they are labeled as Healthy)
true_class = cells_df[label].fillna(healthy_label)

X, _ = get_X_y_data(df=cells_df, label=label, shuffle_features=False)
probabilities = model.predict_proba(X[model.feature_names_in_])

probabilities_df = pd.concat(
    [
        cells_df[metadata_cols + ["dataset"]],
        pd.DataFrame(
            {
                "true_class": true_class,
                "predicted_class": class_names[probabilities.argmax(axis=1)],
            }
        ),
        pd.DataFrame(probabilities, columns=[f"probability_{name}" for name in class_names]),
    ],
    axis=1,
)

print(probabilities_df.shape)
probabilities_df.head()

(10612, 15)


,Metadata_Plate,Metadata_Well,Metadata_Site,Metadata_Nuclei_Number_Object_Number,Metadata_heart_number,Metadata_cell_type,Metadata_heart_failure_type,Metadata_treatment,dataset,true_class,predicted_class,probability_DCM,probability_HLHS,probability_Healthy,probability_ICM
0,CARD-CelIns-CX7_260407120001,E04,17,4,7,Healthy,None,None,train,Healthy,Healthy,2.547160e-01,0.083765,4.082584e-01,0.253261
1,CARD-CelIns-CX7_260407120001,B04,15,5,38,Healthy,None,None,train,Healthy,Healthy,3.589140e-02,0.000639,6.148040e-01,0.348666
2,CARD-CelIns-CX7_260407120001,G10,22,2,36,Failing,DCM,None,train,DCM,DCM,5.660108e-01,0.000917,3.163639e-01,0.116709
3,CARD-CelIns-CX7_260407120001,B09,24,2,2,Healthy,None,None,train,Healthy,Healthy,3.055964e-01,0.000218,6.547457e-01,0.039440
4,CARD-CelIns-CX7_260407120001,F04,19,18,A,Failing,HLHS,None,train,HLHS,HLHS,3.822854e-09,0.999925,2.489389e-11,0.000075


## Check that every cell matches one cell in the normalized profiles

Each cell should be in one split only, and every cell in the normalized profiles should have a prediction.

In [4]:
normalized_ids_df = pd.read_parquet(normalized_path, columns=cell_id_cols)

assert not probabilities_df.duplicated(subset=cell_id_cols).any()
merged_df = normalized_ids_df.merge(
    probabilities_df[cell_id_cols], on=cell_id_cols, how="outer", validate="one_to_one", indicator=True
)
assert (merged_df["_merge"] == "both").all()

pd.DataFrame(
    {
        "n_cells_normalized": [normalized_ids_df.shape[0]],
        "n_cells_with_predictions": [probabilities_df.shape[0]],
    }
)

,n_cells_normalized,n_cells_with_predictions
0,10612,10612


## Summarize and save the predicted probabilities

In [5]:
# Mean predicted probability per split and class
display(
    probabilities_df.groupby(["dataset", "true_class"])[
        [f"probability_{name}" for name in class_names]
    ]
    .mean()
    .round(3)
    .assign(n_cells=probabilities_df.groupby(["dataset", "true_class"]).size())
)

output_path = performance_metrics_dir / "multiclass_single_cell_predicted_probabilities.parquet"
probabilities_df.to_parquet(output_path, index=False)

pd.DataFrame({"output_file": [str(output_path)], "n_cells": [probabilities_df.shape[0]]})

probability_DCM  probability_HLHS  \
dataset      true_class                                      
holdout_well DCM                   0.556             0.037   
             HLHS                  0.003             0.935   
             Healthy               0.205             0.002   
             ICM                   0.210             0.121   
test         DCM                   0.613             0.021   
             HLHS                  0.018             0.922   
             Healthy               0.179             0.005   
             ICM                   0.223             0.085   
train        DCM                   0.660             0.018   
             HLHS                  0.015             0.930   
             Healthy               0.130             0.003   
             ICM                   0.184             0.060   

                         probability_Healthy  probability_ICM  n_cells  
dataset      true_class                                                 
holdout_well DCM                       0.214            0.194      450  
             HLHS                      0.000            0.062      745  
             Healthy                   0.687            0.106      350  
             ICM                       0.116            0.554      531  
test         DCM                       0.157            0.209      565  
             HLHS                      0.002            0.058     1029  
             Healthy                   0.702            0.114      415  
             ICM                       0.121            0.571      552  
train        DCM                       0.137            0.185     1318  
             HLHS                      0.002            0.054     2401  
             Healthy                   0.784            0.082      968  
             ICM                       0.113            0.643     1288

,output_file,n_cells
0,performance_metrics/CARD-CelIns-CX7_2604071200...,10612
